In [1]:
import os
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm
 
from scipy.stats import norm
from scipy.optimize import minimize

In [2]:
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
vars_keep_input = [
    'Fairbanks_T2_1d_max_ensmean', 'Fairbanks_T2_1h_max_ensmean', 'Fairbanks_T2_30d_max_ensmean', 'Fairbanks_T2_7d_max_ensmean', 
    'Fort_Bragg_PRECT_1d_max_ensmean', 'Fort_Bragg_PRECT_3d_max_ensmean', 'Fort_Bragg_PRECT_5d_max_ensmean', 
    'Fort_Bragg_SPEI_09_ensmean_mean', 'Fort_Bragg_SPEI_09_ensmean_min', 'Fort_Bragg_SPEI_48_ensmean_mean', 'Fort_Bragg_SPEI_48_ensmean_min', 
    'Guam_SPEI_03_ensmean_mean', 'Guam_SPEI_03_ensmean_min', 'Guam_SPEI_48_ensmean_mean', 'Guam_SPEI_48_ensmean_min',
    'Pituffik_FT_days_ensmean', 'Pituffik_MDD_ensmean', 
    'Yuma_PG_PRECT_1d_max_ensmean', 'Yuma_PG_PRECT_7d_max_ensmean', 
    'Yuma_PG_SPEI_24_ensmean_mean', 'Yuma_PG_SPEI_24_ensmean_min', 'Yuma_PG_SPEI_48_ensmean_mean', 'Yuma_PG_SPEI_48_ensmean_min', 'Yuma_PG_days_above_999p_ensmean'
]

vars_keep_target = [
    'Fairbanks_T2_1d_max', 'Fairbanks_T2_1h_max', 'Fairbanks_T2_30d_max', 'Fairbanks_T2_7d_max', 
    'Fort_Bragg_PRECT_1d_max', 'Fort_Bragg_PRECT_3d_max', 'Fort_Bragg_PRECT_5d_max', 'Fort_Bragg_SPEI_09_mean', 'Fort_Bragg_SPEI_09_min', 'Fort_Bragg_SPEI_48_mean', 'Fort_Bragg_SPEI_48_min', 
    'Guam_SPEI_03_mean', 'Guam_SPEI_03_min', 'Guam_SPEI_48_mean', 'Guam_SPEI_48_min', 
    'Pituffik_FT_days', 'Pituffik_MDD', 
    'Yuma_PG_PRECT_1d_max', 'Yuma_PG_PRECT_7d_max', 'Yuma_PG_SPEI_24_mean', 'Yuma_PG_SPEI_24_min', 'Yuma_PG_SPEI_48_mean', 'Yuma_PG_SPEI_48_min', 'Yuma_PG_days_above_999p'
]

rename_dict = dict(zip(vars_keep_input, vars_keep_target))

In [4]:
# verified variables
vars_verif = [
    'Fairbanks_T2_1d_max',
    'Fairbanks_T2_1h_max',
    'Fairbanks_T2_30d_max',
    'Fairbanks_T2_7d_max',
    'Fort_Bragg_PRECT_1d_max',
    'Fort_Bragg_PRECT_3d_max',
    'Fort_Bragg_PRECT_5d_max',
    'Fort_Bragg_SPEI_09_mean',
    'Fort_Bragg_SPEI_09_min',
    'Fort_Bragg_SPEI_48_mean',
    'Fort_Bragg_SPEI_48_min',
    'Guam_SPEI_03_mean',
    'Guam_SPEI_03_min',
    'Guam_SPEI_48_mean',
    'Guam_SPEI_48_min',
    'Pituffik_FT_days',
    'Pituffik_MDD',
    'Yuma_PG_PRECT_1d_max',
    'Yuma_PG_PRECT_7d_max',
    'Yuma_PG_SPEI_24_mean',
    'Yuma_PG_SPEI_24_min',
    'Yuma_PG_SPEI_48_mean',
    'Yuma_PG_SPEI_48_min',
]

years_verif = np.arange(2000, 2026)
years_train = np.arange(1959, 2000)
data_dir = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/'
save_dir = '/glade/derecho/scratch/ksha/EPRI_data/PP_calib/AnEn/'

ds_target = xr.open_zarr(os.path.join(data_dir, 'STN_ERA5_ALL.zarr'))[vars_verif]

# ds_target_hist for determine percentiles
ds_target_hist = ds_target.sel(valid_year=years_train)

# ds_target with dims of `(valid_year: 26)`, valid_year = init_year + lead_year, up to 2025
ds_target = ds_target.sel(valid_year=years_verif)[vars_verif]

ds_dress_all = ds_dress = xr.open_zarr(save_dir+'Dress.zarr').rename(rename_dict)[vars_verif]

ds_dress_mean = ds_dress.mean('member')
ds_dress_mean = ds_dress_mean.rename({v: f"{v}_mu" for v in ds_dress_mean.data_vars})

ds_dress_std = ds_dress.std('member')
ds_dress_std = ds_dress_std.rename({v: f"{v}_sigma" for v in ds_dress_std.data_vars})

ds_calib_all = xr.merge([ds_dress_mean, ds_dress_std])

### Verification Scores

In [6]:
# ============================================================================
# 4. ACC and CRPS per (variable, lead_year)
# ============================================================================
def gather_pairs_at_lead(da_fc, da_obs, il, has_member=True):
    """Forecast/obs pairs for a single lead_year index `il`.
    Returns fc shape (N, M) or (N,), obs shape (N,)."""
    init_years = da_fc['init_year'].values
    lead       = int(da_fc['lead_year'].values[il])
    valid_yrs  = init_years + lead

    obs_avail = da_obs['valid_year'].values
    in_mask   = np.isin(valid_yrs, obs_avail)
    if not in_mask.any():
        return (np.empty((0, da_fc.sizes['member'])) if has_member
                else np.empty(0)), np.empty(0)

    obs_arr = da_obs.sel(valid_year=valid_yrs[in_mask]).values

    if has_member:
        fc_arr = da_fc.isel(lead_year=il).values[in_mask, :]
        finite = np.all(np.isfinite(fc_arr), axis=1) & np.isfinite(obs_arr)
        return fc_arr[finite, :], obs_arr[finite]
    else:
        fc_arr = da_fc.isel(lead_year=il).values[in_mask]
        finite = np.isfinite(fc_arr) & np.isfinite(obs_arr)
        return fc_arr[finite], obs_arr[finite]


def crps_ensemble_pooled(ens, obs):
    if ens.size == 0:
        return np.nan
    term1 = np.mean(np.abs(ens - obs[:, None]), axis=1)
    diff  = np.abs(ens[:, :, None] - ens[:, None, :])
    term2 = 0.5 * diff.mean(axis=(1, 2))
    return float(np.mean(term1 - term2))


def acc_pooled(fc_mean, obs, clim):
    if fc_mean.size < 2 or not np.isfinite(clim):
        return np.nan
    fa = fc_mean - clim
    oa = obs     - clim
    den = np.sqrt(np.sum(fa * fa) * np.sum(oa * oa))
    return float(np.sum(fa * oa) / den) if den > 0 else np.nan


lead_years = ds_dress_all['lead_year'].values
n_lead = len(lead_years)
n_var  = len(vars_verif)

crps_arr    = np.full((n_var, n_lead), np.nan)
acc_calib   = np.full((n_var, n_lead), np.nan)
acc_dressmn = np.full((n_var, n_lead), np.nan)
n_eff_arr   = np.zeros((n_var, n_lead), dtype=np.int64)
clim_arr    = np.full(n_var, np.nan)
bias_arr = np.full((n_var, n_lead), np.nan)

for iv, var in enumerate(tqdm(vars_verif, desc='ACC / CRPS     ')):
    # Climatology (single value per variable, from 1959-1999 ERA5)
    hist = ds_target_hist[var].values
    hist = hist[np.isfinite(hist)]
    clim = float(hist.mean()) if hist.size else np.nan
    clim_arr[iv] = clim

    for il in range(n_lead):
        # CRPS from dressed ensemble
        fc_ens, obs = gather_pairs_at_lead(ds_dress_all[var],
                                           ds_target[var], il, has_member=True)
        crps_arr[iv, il]  = crps_ensemble_pooled(fc_ens, obs)
        n_eff_arr[iv, il] = int(obs.size)

        # ACC from ds_calib_all _mu
        mu_flat, obs_mu = gather_pairs_at_lead(ds_calib_all[f'{var}_mu'],
                                               ds_target[var], il, has_member=False)
        acc_calib[iv, il] = acc_pooled(mu_flat, obs_mu, clim)

        # ACC from dressed ensemble mean (sanity check)
        if fc_ens.size:
            acc_dressmn[iv, il] = acc_pooled(fc_ens.mean(axis=1), obs, clim)
            bias_arr[iv, il] = float(fc_ens.mean(axis=1).mean() - obs.mean())


ds_verif = xr.Dataset(
    {
        'CRPS':           (('variable', 'lead_year'), crps_arr),
        'ACC_calib':      (('variable', 'lead_year'), acc_calib),
        'ACC_dress_mean': (('variable', 'lead_year'), acc_dressmn),
        'n_pairs':        (('variable', 'lead_year'), n_eff_arr),
        'climatology':    (('variable',),             clim_arr),
        'Bias':           (('variable', 'lead_year'), bias_arr),
    },
    coords={'variable': vars_verif, 'lead_year': lead_years},
)
ds_verif['CRPS'].attrs['description']           = 'Empirical CRPS of 50-member dressed ensemble vs ERA5, per lead_year'
ds_verif['ACC_calib'].attrs['description']      = 'ACC of calibrated mean (_mu) vs ERA5, climatology from 1959-1999, per lead_year'
ds_verif['ACC_dress_mean'].attrs['description'] = 'ACC of dressed ensemble mean vs ERA5, per lead_year'
ds_verif['climatology'].attrs['description']    = 'ERA5 1959-1999 mean used as ACC reference'
ds_verif['Bias'].attrs['description'] = 'Mean bias: mean(ensemble_mean) - mean(obs), per lead_year'

ds_verif.to_zarr(save_dir+'/verif_scores.zarr', mode='w')
print(save_dir+'/verif_scores.zarr')

ACC / CRPS     : 100%|██████████| 23/23 [00:04<00:00,  5.67it/s]

/glade/derecho/scratch/ksha/EPRI_data/PP_calib/AnEn//verif_scores.zarr


In [7]:
# Quick summary
print("\nPer-lead CRPS:")
print(ds_verif['CRPS'].to_pandas().round(3))
print("\nPer-lead ACC_calib:")
print(ds_verif['ACC_calib'].to_pandas().round(3))


Per-lead CRPS:
lead_year                     0       1       2       3       4       5  \
variable                                                                  
Fairbanks_T2_1d_max       0.831   0.849   0.821   0.848   0.847   0.823   
Fairbanks_T2_1h_max       1.177   1.089   1.088   1.222   1.088   1.034   
Fairbanks_T2_30d_max      0.800   0.820   0.837   0.841   0.838   0.800   
Fairbanks_T2_7d_max       0.772   0.813   0.777   0.793   0.797   0.815   
Fort_Bragg_PRECT_1d_max  17.553  16.193  18.199  19.128  23.362  20.255   
Fort_Bragg_PRECT_3d_max   9.974   9.671   9.623   9.698  11.390  10.894   
Fort_Bragg_PRECT_5d_max   6.556   6.360   6.167   6.299   7.056   6.740   
Fort_Bragg_SPEI_09_mean   0.512   0.547   0.529   0.529   0.505   0.540   
Fort_Bragg_SPEI_09_min    0.611   0.614   0.617   0.589   0.607   0.583   
Fort_Bragg_SPEI_48_mean   0.882   1.087   0.958   0.943   1.041   1.101   
Fort_Bragg_SPEI_48_min    1.023   1.225   1.069   1.072   1.190   1.265   
Guam_SPEI